# World Cup 2026 Predictor — Python + SQL

Companion notebook to `r_sql_predictor.Rmd`. Both notebooks read from the same SQLite database (`data/wc2026.sqlite`) built by `src/build_db.py`, and both fit the same Dixon–Coles bivariate Poisson model. Running them side-by-side gives a cross-language sanity check.

**Pipeline**
1. Connect to SQLite and inspect the source tables
2. Engineer a strength score per team (EA FC squad overall × FIFA points)
3. Fit Dixon–Coles on time-decayed historical international matches
4. Layer corners / cards baseline and the shootout model
5. Monte‐Carlo the group stage (10 000 runs), resolve the bracket
6. Write the predictions back to SQLite and plot the headline results

In [ ]:
import sys, sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src import models as M

DB = ROOT / 'data' / 'wc2026.sqlite'
conn = sqlite3.connect(DB)
print('Connected to', DB)

## 1 Inspect the source tables

In [ ]:
tables = ['historical_matches', 'fifa_rankings', 'player_ratings',
          'team_squad_strength', 'wc2026_groups', 'wc2026_fixtures']
rows = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tables}
pd.Series(rows, name='rows').to_frame()

In [ ]:
pd.read_sql('SELECT * FROM wc2026_groups ORDER BY group_id, pot', conn).head(20)

## 2 Team-strength feature

Blend the EA FC 26 top-11 squad rating (current talent signal) with the latest FIFA ranking points (recent-form signal). EA leans heavier (60/40) because it incorporates club-level performance for each player.

In [ ]:
strength = pd.read_sql('''
    WITH latest_rank AS (
        SELECT team, rank, points,
               ROW_NUMBER() OVER (PARTITION BY team ORDER BY rank_date DESC) AS rn
        FROM fifa_rankings
    )
    SELECT g.team, s.squad_overall_top11, s.squad_attack_mean, s.squad_defense_mean,
           r.rank AS fifa_rank, r.points AS fifa_points
    FROM wc2026_groups g
    LEFT JOIN team_squad_strength s ON s.team = g.team
    LEFT JOIN latest_rank r ON r.team = g.team AND r.rn = 1
''', conn)

def z(x):
    return (x - x.mean()) / x.std(ddof=0)

strength['z_squad'] = z(strength['squad_overall_top11'])
strength['z_fifa']  = z(strength['fifa_points'])
strength['strength'] = 0.6 * strength['z_squad'].fillna(0) + 0.4 * strength['z_fifa'].fillna(0)
strength.sort_values('strength', ascending=False).head(15)

## 3 Fit Dixon-Coles

Train on international matches from 2014 onwards with exponential time-decay (ξ = 0.0019/day, the value Dixon–Coles propose for football).

In [ ]:
hist = pd.read_sql('''
    SELECT match_date, home_team, away_team, home_score, away_score, neutral
    FROM historical_matches
    WHERE match_date >= '2014-01-01'
      AND home_score IS NOT NULL AND away_score IS NOT NULL
''', conn)
hist['match_date'] = pd.to_datetime(hist['match_date'])
hist['neutral'] = hist['neutral'].fillna(0).astype(int)
len(hist)

In [ ]:
model = M.DixonColesModel()
model.fit(hist, ref_date=pd.Timestamp('2026-06-10'))
print(f'Fitted: {len(model.teams)} teams, home_adv={model.home_adv:.3f}, rho={model.rho:.3f}')

params = pd.DataFrame({
    'team': model.teams,
    'attack': [model.attack[t] for t in model.teams],
    'defense': [model.defense[t] for t in model.teams],
})
params.sort_values('attack', ascending=False).head(15)

### Sanity check — sample match score matrix

In [ ]:
sm = model.score_matrix('Brazil', 'Scotland', neutral=True, max_goals=6)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sm, cmap='Blues', origin='lower')
for i in range(sm.shape[0]):
    for j in range(sm.shape[1]):
        ax.text(j, i, f'{sm[i, j]*100:.1f}', ha='center', va='center',
                color='white' if sm[i, j] > 0.06 else 'black', fontsize=8)
ax.set_xlabel('Scotland goals'); ax.set_ylabel('Brazil goals')
ax.set_title('Brazil vs Scotland — score probability (%)')
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()
print('Modal:', model.modal_score('Brazil', 'Scotland'))
print('P(W/D/L):', tuple(round(p, 3) for p in model.outcome_probs('Brazil', 'Scotland')))

## 4 Group-stage Monte Carlo (10 000 tournaments)

For each simulation we (a) draw a score for every group-stage match, (b) compute the standings, (c) extract the 12 group winners, 12 runners-up and 8 best third-placed teams.

In [ ]:
fixtures_grp = pd.read_sql(
    "SELECT fixture_id, group_id, home_team, away_team "
    "FROM wc2026_fixtures WHERE stage = 'GROUP'", conn)
len(fixtures_grp)

In [ ]:
rng = np.random.default_rng(2026)
N_SIM = 10_000

def standings_from_runs(scores: pd.DataFrame) -> pd.DataFrame:
    home = scores[['group_id', 'home_team', 'h', 'a']].rename(
        columns={'home_team': 'team', 'h': 'gf', 'a': 'ga'})
    away = scores[['group_id', 'away_team', 'a', 'h']].rename(
        columns={'away_team': 'team', 'a': 'gf', 'h': 'ga'})
    long = pd.concat([home, away], ignore_index=True)
    long['pts'] = np.where(long['gf'] > long['ga'], 3,
                  np.where(long['gf'] == long['ga'], 1, 0))
    agg = long.groupby(['group_id', 'team']).agg(
        pts=('pts', 'sum'),
        gd=('gf', lambda s: int(s.sum() - long.loc[s.index, 'ga'].sum())),
        gf=('gf', 'sum'),
    ).reset_index()
    agg['rank'] = agg.groupby('group_id').apply(
        lambda d: d.sort_values(['pts', 'gd', 'gf'], ascending=False)
                   .reset_index().assign(rank=range(1, len(d) + 1))
                   .set_index('index')['rank']).droplevel(0)
    return agg

champion_counts = {}
reach_round = {}    # team -> dict of round -> count

for sim in range(N_SIM):
    rows = []
    for _, fx in fixtures_grp.iterrows():
        r = M.simulate_match(model, fx['home_team'], fx['away_team'],
                             stage='GROUP', neutral=True, rng=rng)
        rows.append({'group_id': fx['group_id'],
                     'home_team': fx['home_team'],
                     'away_team': fx['away_team'],
                     'h': r['home_score'], 'a': r['away_score']})
    scores = pd.DataFrame(rows)
    st = standings_from_runs(scores)
    top2 = st[st['rank'] <= 2]
    thirds = st[st['rank'] == 3].sort_values(
        ['pts', 'gd', 'gf'], ascending=False).head(8)
    advancers = pd.concat([top2, thirds.assign(rank=3)], ignore_index=True)

    # Simple knockout: pair sorted advancers, play one match per round.
    bracket = advancers.sort_values(['rank', 'group_id'])['team'].tolist()
    for stage in ['R32', 'R16', 'QF', 'SF', 'FINAL']:
        next_round = []
        for i in range(0, len(bracket), 2):
            h, a = bracket[i], bracket[i + 1]
            r = M.simulate_match(model, h, a, stage=stage,
                                 neutral=True, knockout=True, rng=rng)
            winner = h if r['home_advances'] else a
            next_round.append(winner)
            for team in (h, a):
                reach_round.setdefault(team, {}).setdefault(stage, 0)
                reach_round[team][stage] += 1
        bracket = next_round
    champ = bracket[0]
    champion_counts[champ] = champion_counts.get(champ, 0) + 1
    if sim % 500 == 0:
        print(f'sim {sim:>5}/{N_SIM}')

champ_df = (pd.Series(champion_counts, name='wins').to_frame()
            .assign(p_champion=lambda d: d['wins'] / N_SIM)
            .sort_values('p_champion', ascending=False))
champ_df.head(15)

## 5 Headline chart — championship probabilities

In [ ]:
top = champ_df.head(12)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top.index[::-1], top['p_champion'][::-1] * 100,
        color='#1f4e79', edgecolor='white')
ax.set_xlabel('Probability of winning the World Cup (%)')
ax.set_title('World Cup 2026 — championship odds (10 000 simulations)')
for i, v in enumerate(top['p_champion'][::-1] * 100):
    ax.text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()

## 6 Write per-match predictions back to the database

In [ ]:
from datetime import datetime

rows = []
for _, fx in fixtures_grp.iterrows():
    h, a = fx['home_team'], fx['away_team']
    mh, ma = model.modal_score(h, a)
    pw_h, pw_d, pw_a = model.outcome_probs(h, a)
    sg = (model.attack.get(h, 0) - model.attack.get(a, 0)
          + model.defense.get(a, 0) - model.defense.get(h, 0))
    ch, ca = M.corners_baseline('GROUP', sg)
    yh, ya, prh, pra = M.cards_baseline('GROUP', sg)
    rows.append({
        'fixture_id': int(fx['fixture_id']), 'source': 'python',
        'modal_home_score': mh, 'modal_away_score': ma,
        'p_home_win': pw_h, 'p_draw': pw_d, 'p_away_win': pw_a,
        'exp_home_corners': ch, 'exp_away_corners': ca,
        'exp_home_yellows': yh, 'exp_away_yellows': ya,
        'p_home_red': prh, 'p_away_red': pra,
        'p_penalties': None, 'p_home_advances': None,
        'generated_at': datetime.now().isoformat(timespec='seconds'),
    })

preds = pd.DataFrame(rows)
conn.execute("DELETE FROM predictions WHERE source = 'python'")
preds.to_sql('predictions', conn, if_exists='append', index=False)
conn.commit()
print(f'Wrote {len(preds)} predictions to SQLite')
preds.head()

In [ ]:
# Persist tournament-level Monte Carlo results too
sim_rows = []
for team in strength['team']:
    sim_rows.append({
        'team': team, 'source': 'python',
        'p_champion': champion_counts.get(team, 0) / N_SIM,
        'p_reach_final': reach_round.get(team, {}).get('FINAL', 0) / N_SIM,
        'p_reach_sf':    reach_round.get(team, {}).get('SF', 0) / N_SIM,
        'p_reach_qf':    reach_round.get(team, {}).get('QF', 0) / N_SIM,
        'p_reach_r16':   reach_round.get(team, {}).get('R16', 0) / N_SIM,
        'p_advance_r32': reach_round.get(team, {}).get('R32', 0) / N_SIM,
        'n_simulations': N_SIM,
    })
sim_df = pd.DataFrame(sim_rows)
conn.execute("DELETE FROM tournament_sim WHERE source = 'python'")
sim_df.to_sql('tournament_sim', conn, if_exists='append', index=False)
conn.commit()
sim_df.sort_values('p_champion', ascending=False).head(10)

In [ ]:
conn.close()
print('Done.')